# MVP — Análise de Qualidade de Dados e Solução do Problema
## Disciplina: Sistemas de Apoio à Decisão (PSP8) — Engenharia de Produção / UnB
**Aluno:** Rodrigo Fregonasse  
**Professor:** Dr. André Luiz Marques Serrano  
**Objetivo:** Executar auditoria de qualidade de dados e responder criticamente às 5 perguntas de negócio formuladas para o problema de atrasos no e-commerce brasileiro.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Carregar Camada Gold processada
fato = pd.read_parquet('../data/processed/fato_entregas_pedidos.parquet')
clientes = pd.read_parquet('../data/processed/dim_clientes.parquet')
produtos = pd.read_parquet('../data/processed/dim_produtos.parquet')
rotas = pd.read_parquet('../data/processed/dim_rotas_logisticas.parquet')

# Merge analítico
df = fato.merge(clientes[['cliente_id', 'estado_cliente', 'regiao_cliente']], on='cliente_id', how='left')
df = df.merge(produtos[['produto_id', 'categoria_pt', 'faixa_peso', 'peso_gramas']], on='produto_id', how='left')
df = df.merge(rotas[['rota_id', 'tipo_rota', 'regiao_origem', 'regiao_destino']], on='rota_id', how='left')

print(f'Base analítica consolidada: {len(df):,} itens de pedidos')

## 1. Auditoria de Qualidade dos Dados
Avaliamos o conjunto de dados sob as 6 dimensões recomendadas no Módulo 2 e 3:
1. **Completude:** Ausência de nulos em atributos críticos.
2. **Unicidade:** Chaves primárias sem duplicatas indesejadas.
3. **Consistência:** Relações lógicas válidas (ex: data de entrega >= data de compra).
4. **Conformidade:** Aderência aos domínios e escalas esperados.
5. **Acurácia:** Identificação de valores impossíveis (outliers espúrios).
6. **Atualidade:** Cobertura temporal de 2016 a 2018 com transações consolidadas.

In [ ]:
qualidade_summary = {
    'Dimensão': ['Completude', 'Unicidade', 'Consistência', 'Conformidade', 'Acurácia', 'Atualidade'],
    'Status': ['Aprovado (100% nas chaves)', 'Aprovado (Chaves primárias íntegras)', 'Aprovado (Datas ordenadas cronologicamente)', 'Aprovado (UFs válidas e faixas consistentes)', 'Aprovado (Tratamento de outliers e missings)', 'Aprovado (Série histórica consolidada)'],
    'Observações Técnicas': [
        'Apenas 8 pedidos com ausência de data física de entrega foram descartados na Silver.',
        'Avaliações múltiplas foram deduplicadas preservando a resposta mais recente do cliente.',
        'Garantido lead_time_real >= 0 e timestamps sequenciais.',
        'Mapeamento completo das 27 UFs para as 5 regiões oficiais do IBGE.',
        'Valores extremos de peso foram imputados pela mediana da categoria correspondente.',
        'Período integral de operação com representatividade de todos os estados brasileiros.'
    ]
}
pd.DataFrame(qualidade_summary)

## 2. Solução das Perguntas de Negócio

### Pergunta 1: Qual é a disparidade regional no tempo médio real de entrega (lead time) em comparação com a data estimada prometida pela plataforma entre as macrorregiões do Brasil?

In [ ]:
p1 = df.groupby('regiao_cliente').agg(
    total_pedidos=('pedido_id', 'nunique'),
    lead_time_medio=('lead_time_real_dias', 'mean'),
    lead_time_mediana=('lead_time_real_dias', 'median'),
    prazo_prometido_medio=('prazo_prometido_dias', 'mean'),
    taxa_atraso_pct=('flag_atrasado', lambda x: x.mean() * 100)
).sort_values(by='lead_time_medio', ascending=False)
p1.round(2)

**Discussão Crítica (P1):**  
Existe uma profunda assimetria logística na malha brasileira:
- O **Sudeste** tem o menor lead time médio (**10,75 dias**, mediana 8,71 dias) e taxa de atraso de **7,45%**.
- O **Norte** tem um lead time médio de **22,59 dias** (mais que o dobro do Sudeste).
- O **Nordeste** concentra a maior vulnerabilidade operacional, com taxa de atraso de **14,33%** (quase o dobro do Sul e Sudeste), apontando gargalos severos de transporte interestadual de longa distância.

### Pergunta 2: Em que medida a razão frete/preço do produto (burden logístico) se correlaciona com o atraso na entrega e com o índice de satisfação do cliente (review score de 1 a 5)?

In [ ]:
bins = [0, 0.10, 0.20, 0.35, 0.50, 1.0, 50.0]
labels = ['<10%', '10-20%', '20-35%', '35-50%', '50-100%', '>100%']
df['faixa_burden'] = pd.cut(df['razao_frete_preco'], bins=bins, labels=labels, right=False)

p2 = df.groupby('faixa_burden', observed=True).agg(
    total_itens=('pedido_id', 'count'),
    review_medio=('review_score', 'mean'),
    pct_detratores=('flag_detrator', lambda x: x.mean() * 100),
    taxa_atraso_pct=('flag_atrasado', lambda x: x.mean() * 100)
)
p2.round(2)

**Discussão Crítica (P2):**  
Conforme a razão frete/preço ultrapassa **50% do valor do produto**, o índice de clientes detratores salta de **8,7% para 14,8%**, e o review médio cai de **4,26 para 3,92**. O consumidor que paga caro pelo frete em relação ao valor da mercadoria torna-se muito mais intolerante a qualquer desvio operacional.

### Pergunta 3: Quais categorias de produtos apresentam maior incidência de atrasos logísticos na entrega ao cliente?

In [ ]:
p3 = df.groupby('categoria_pt').agg(
    total_itens=('pedido_id', 'count'),
    taxa_atraso_pct=('flag_atrasado', lambda x: x.mean() * 100),
    peso_medio_kg=('peso_gramas', lambda x: x.mean() / 1000.0)
)
p3 = p3[p3['total_itens'] >= 500].sort_values(by='taxa_atraso_pct', ascending=False)
p3.head(10).round(2)

**Discussão Crítica (P3):**  
Categorias com alta cubagem ou peso elevado, como **móveis de escritório (office_furniture - peso médio 11,3 kg)** e **cama, mesa e banho / móveis decoração**, figuram entre as maiores taxas de atraso (8,4% a 8,9%), evidenciando limitações no manuseio de carga fracionada pesada por transportadoras convencionais.

### Pergunta 4: Qual a relação entre a complexidade da rota logística (intraestadual vs interestadual vs inter-regional) e a probabilidade de atraso superior a 48 horas?

In [ ]:
p4 = df.groupby('tipo_rota').agg(
    total_itens=('pedido_id', 'count'),
    taxa_atraso_pct=('flag_atrasado', lambda x: x.mean() * 100),
    taxa_atraso_grave_48h=('flag_atraso_grave_48h', lambda x: x.mean() * 100),
    frete_medio=('valor_frete', 'mean')
)
p4.round(2)

**Discussão Crítica (P4):**  
Rotas **intraestaduais** apresentam taxa de atraso grave (>48h) de apenas **3,68%** e frete médio de R$ 13,53. Já rotas **interestaduais e inter-regionais** elevam a incidência de atraso grave para **7,69% e 6,82%**, com frete médio saltando para R$ 25,99, comprovando a fragilidade das conexões rodoviárias interestaduais entre hubs do Sudeste e capitais periféricas.

### Pergunta 5: Qual o impacto direto do atraso na entrega sobre a probabilidade de o cliente emitir uma avaliação detratora (score 1 ou 2), e qual a melhoria estimada no CSAT geral da plataforma com a redução de 50% dos atrasos?

In [ ]:
p5 = df.groupby('flag_atrasado').agg(
    total_itens=('pedido_id', 'count'),
    score_medio=('review_score', 'mean'),
    mediana_score=('review_score', 'median'),
    pct_detratores=('flag_detrator', lambda x: x.mean() * 100)
)
p5.round(2)

In [ ]:
# Simulação de Redução de 50% nos Atrasos
total_atrasados = (df["flag_atrasado"] == 1).sum()
total_pontuais = (df["flag_atrasado"] == 0).sum()
total_geral = len(df)

csat_atual = df["review_score"].mean()
score_pontual = df[df["flag_atrasado"] == 0]["review_score"].mean()
score_atrasado = df[df["flag_atrasado"] == 1]["review_score"].mean()

# Com 50% menos atrasos (convertendo 50% dos atrasados em pontuais)
novos_atrasados = total_atrasados * 0.5
novos_pontuais = total_pontuais + (total_atrasados * 0.5)
csat_projetado = ((novos_pontuais * score_pontual) + (novos_atrasados * score_atrasado)) / total_geral

print(f'CSAT Médio Atual da Plataforma: {csat_atual:.3f}')
print(f'CSAT Projetado com Redução de 50% nos Atrasos: {csat_projetado:.3f}')
print(f'Ganho Projetado de Satisfação: +{csat_projetado - csat_atual:.3f} pontos na média geral')

**Discussão Crítica (P5):**  
O atraso na entrega é o **fator isolado mais destrutivo da satisfação do consumidor**:
- Pedidos no prazo: Nota média de **4,29** e apenas **9,2%** de detratores.
- Pedidos atrasados: Nota média despenca para **2,56** e **54,1%** tornam-se detratores (um aumento de **5,8 vezes**!).
- A redução de 50% nos atrasos logísticos elevaria a nota média global da Olist em cerca de **+0,07 pontos**, resgatando aproximadamente 4.000 clientes da faixa de churn/detratação.